# QSTD — a first look at the dataset

The **Quantum Sensor-aligned Telemetry Dataset**: 1,549,234 quantum circuit
executions on two 20-qubit computers, each carrying how well it ran, the circuit
that ran, the facility sensor readings from the minutes before it finished, and
the device's most recent calibration.

This notebook **only looks at the data**. It trains nothing — the models behind
the paper live in `experiments/` and are run by `scripts/reproduce_all.sh`.
Keeping the two apart means a figure here can never quietly disagree with a table
there.

**Before you start:** download `qstd_v1.0.parquet` from the portal into `data/`.
Nothing else is needed; there is no conversion step.

```bash
uv run jupyter lab analysis/qstd_overview.ipynb
```

Run time: about two minutes.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()

# The release ships as qstd_v1.0.parquet; a build tree calls it stage 10.
for name in ("qstd_v1.0.parquet", "qstd.parquet", "stage10_cleaned_final_dataset.parquet"):
    DATA = REPO / "data" / name
    if DATA.exists():
        break
else:
    raise FileNotFoundError(
        "No dataset in data/. Download qstd_v1.0.parquet from the portal "
        "(see data/README.md) and put it there."
    )

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7.5, 3.6),
                     "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9})
SC, ION = "superconducting_20q", "trapped_ion_20q"
COLOUR = {SC: "#2b6cb0", ION: "#c05621"}
NAME = {SC: "superconducting, 20 qubits", ION: "trapped ion, 20 qubits"}

df = pd.read_parquet(DATA)
df["month"] = pd.to_datetime(df["completed_hour_utc"], utc=True).dt.tz_convert(None).dt.to_period("M")
print(f"{DATA.name}: {len(df):,} records x {len(df.columns)} columns")
print(f"period: {df.month.min()} to {df.month.max()}")

## 1. What is in a record

One record is one circuit execution. The columns fall into seven groups.

In [ ]:
def group_of(c):
    if c in ("job_id", "circuit_id", "device"):                return "identifier"
    if c.endswith("_hour_utc"):                                 return "time"
    if c in ("status", "batch_size", "shots"):                  return "job"
    if c.startswith("hellinger_"):                              return "target"
    if c.startswith("sc_cal_"):                                 return "calibration"
    if c.startswith(("sc_cryo_", "sc_cooling_", "ion_lab_")):   return "sensor"
    if c.startswith(("logical_", "native_")):                   return "circuit"
    return "other"

groups = pd.Series({c: group_of(c) for c in df.columns}).value_counts()
print(groups.to_string())
print(f"total: {len(df.columns)} columns")
df[["job_id", "circuit_id", "device", "completed_hour_utc", "shots",
    "native_n_2q_gates", "hellinger_native"]].head()

## 2. The target

`hellinger_native` is the distance between the measured outcome distribution and
a noise-free simulation of the same circuit at the same shot count. **0 means
identical, 1 means no overlap.** `hellinger_logical` is the same against the
circuit as submitted; the two correlate at r = 0.999, so use one or the other.

The distribution is bimodal — circuits either run well or collapse — and the two
devices sit in different places.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for dev in (SC, ION):
    s = df.loc[df.device == dev, "hellinger_native"].dropna()
    axes[0].hist(s, bins=80, alpha=0.6, label=f"{NAME[dev]}  (n={len(s):,})", color=COLOUR[dev])
axes[0].set_xlabel("hellinger_native"); axes[0].set_ylabel("records")
axes[0].set_title("Target distribution"); axes[0].legend(fontsize=8)

data = [df.loc[df.device == d, "hellinger_native"].dropna() for d in (SC, ION)]
axes[1].boxplot(data, labels=["superconducting", "trapped ion"], showfliers=False)
axes[1].set_ylabel("hellinger_native"); axes[1].set_title("By device")
plt.tight_layout(); plt.show()

print(df.groupby("device")["hellinger_native"].describe()[["count","mean","50%","std"]].to_string())

## 3. Shot count is a first-order driver

Sampling noise scales as 1/sqrt(shots), so the shot count puts a **floor** under
the distance: a 200-shot circuit cannot score below about 0.07 however well it
ran. Different shot counts are different workloads, and any model trained across
them can separate the buckets instead of predicting fidelity. Control for it.

In [ ]:
sh = pd.to_numeric(df["shots"], errors="coerce")
top = sh.value_counts().head(8).sort_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar([str(int(i)) for i in top.index], top.values, color="#4a5568")
axes[0].set_xlabel("shots"); axes[0].set_ylabel("records"); axes[0].set_title("Shot buckets")

for s_val in [v for v in (200, 2000, 4000, 10000) if v in set(sh.dropna())]:
    sel = df.loc[sh == s_val, "hellinger_native"].dropna()
    if len(sel) > 500:
        axes[1].hist(sel, bins=60, histtype="step", lw=1.6, label=f"{s_val:,} shots")
axes[1].set_xlabel("hellinger_native"); axes[1].set_title("Target, by shot bucket")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("noise floor 1/sqrt(shots):")
for v in sorted(set(top.index)):
    print(f"  {int(v):>6,} shots -> {1/np.sqrt(v):.4f}   ({int((sh==v).sum()):,} records)")

## 4. Coverage over time

**Empty means not measured, never zero.** Sensor and calibration coverage varies
by month, and this is the single most important thing to understand before
modelling: a model trained across months where the sensors were off can let
*missingness* stand in for the calendar.

In [ ]:
sensor_cols = [c for c in df.columns if c.startswith(("sc_cryo_","sc_cooling_","ion_lab_"))]
calib_cols  = [c for c in df.columns if c.startswith("sc_cal_")]
sc_sensors  = [c for c in sensor_cols if c.startswith(("sc_cryo_","sc_cooling_"))]
ion_sensors = [c for c in sensor_cols if c.startswith("ion_lab_")]

rows = []
for (m, dev), g in df.groupby(["month", "device"]):
    if dev not in (SC, ION):
        continue
    cols = sc_sensors if dev == SC else ion_sensors
    rows.append({"month": str(m), "device": dev, "records": len(g),
                 "sensor_pct": 100 * g[cols].notna().mean(axis=1).mean(),
                 "calib_pct": 100 * g[calib_cols].notna().any(axis=1).mean() if dev == SC else np.nan})
cov = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for dev in (SC, ION):
    d = cov[cov.device == dev]
    axes[0].plot(d.month, d.records, "o-", ms=3, color=COLOUR[dev], label=NAME[dev])
    axes[1].plot(d.month, d.sensor_pct, "o-", ms=3, color=COLOUR[dev], label=f"{NAME[dev]} sensors")
d = cov[cov.device == SC]
axes[1].plot(d.month, d.calib_pct, "s--", ms=3, color="#2f855a", label="superconducting calibration")
axes[0].set_yscale("log"); axes[0].set_ylabel("records (log)"); axes[0].legend(fontsize=8)
axes[0].set_title("Records per month")
axes[1].set_ylabel("% available"); axes[1].set_ylim(-3, 103); axes[1].legend(fontsize=8)
axes[1].set_title("Sensor channels with a reading, and records with calibration")
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

## 5. What predicts the target

The strongest signals are **structural**, not environmental. Measured-bit count
leads: the outcome space is 2^(measured bits), so sampling noise grows with it at
a fixed shot count — the same mechanism as section 3, seen from another angle.

In [ ]:
circuit_cols = [c for c in df.columns if c.startswith(("logical_","native_"))] + ["shots"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, dev in zip(axes, (SC, ION)):
    g = df[df.device == dev]
    num = g[circuit_cols].apply(pd.to_numeric, errors="coerce")
    corr = num.corrwith(g["hellinger_native"], method="spearman").dropna()
    corr = corr.reindex(corr.abs().sort_values(ascending=False).index)[:12][::-1]
    ax.barh(range(len(corr)), corr.values,
            color=["#c05621" if v < 0 else "#2b6cb0" for v in corr.values])
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index, fontsize=8)
    ax.axvline(0, color="black", lw=0.6); ax.set_title(NAME[dev], fontsize=9)
    ax.set_xlabel("Spearman correlation with hellinger_native")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, dev in zip(axes, (SC, ION)):
    g = df[(df.device == dev)].dropna(subset=["native_n_2q_gates", "hellinger_native"])
    g = g.sample(min(40000, len(g)), random_state=42)
    ax.scatter(g.native_n_2q_gates, g.hellinger_native, s=2, alpha=0.08, color=COLOUR[dev])
    q = g.assign(b=pd.qcut(g.native_n_2q_gates, 30, duplicates="drop")).groupby("b", observed=True)
    med = q.agg(x=("native_n_2q_gates","median"), y=("hellinger_native","median"))
    ax.plot(med.x, med.y, "k-", lw=1.8, label="median trend")
    ax.set_xscale("symlog"); ax.set_xlabel("native_n_2q_gates"); ax.set_ylabel("hellinger_native")
    ax.set_title(NAME[dev], fontsize=9); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. The environment

Facility readings come from the **5 minutes ending at the completion timestamp** —
never the submission time, because a job can queue for hours. Each channel
appears as `_mean`, `_min`, `_max` and `_sd` over that window.

Values are published as the monitoring system reported them: many are scaled
integers. `qstd_v1.0_columns.csv` gives each column's unit and how confident that
unit is — `exact`, `inferred` or `unknown`.

In [ ]:
probe = [c for c in ("sc_cryo_mxc_temp_mean","sc_cryo_4k_temp_mean",
                     "sc_cryo_50k_temp_mean","sc_cryo_still_temp_mean") if c in df.columns]
if probe:
    fig, axes = plt.subplots(1, len(probe), figsize=(3.0*len(probe), 3.0))
    for ax, c in zip(np.atleast_1d(axes), probe):
        s = pd.to_numeric(df[c], errors="coerce").dropna()
        ax.hist(s, bins=60, color="#2b6cb0")
        ax.set_title(c.replace("sc_cryo_","").replace("_mean",""), fontsize=9)
        ax.set_xlabel("raw value")
    plt.suptitle("Cryostat stages — raw published values (see the data dictionary for units)",
                 fontsize=9)
    plt.tight_layout(); plt.show()

print("sensor channels with any reading, by device:")
print(f"  superconducting: {df.loc[df.device==SC, sc_sensors].notna().any().sum()} of {len(sc_sensors)}")
print(f"  trapped ion    : {df.loc[df.device==ION, ion_sensors].notna().any().sum()} of {len(ion_sensors)}")

## 7. Known limitations, visible in the data

Worth checking yourself before modelling.

In [ ]:
sh = pd.to_numeric(df["shots"], errors="coerce")
no_target = df.hellinger_native.isna()
print(f"records with no target          : {int(no_target.sum()):,}  "
      f"({100*no_target.mean():.2f}%) - the job result could not be split per circuit")
print(f"records at 1-2 shots            : {int((sh<=2).sum()):,}  "
      f"- their distance can only be 0 or 1, so it carries no information")
print(f"records at <= 100 shots         : {int((sh<=100).sum()):,}")
print(f"records on the two main devices : {int(df.device.isin([SC,ION]).sum()):,}  "
      f"({100*df.device.isin([SC,ION]).mean():.1f}%)")
print()
print("records per device:")
print(df.device.value_counts().to_string())
print()
print("group on job_id, never on circuit_id: a job's circuits ran back to back")
print(f"  jobs: {df.job_id.nunique():,}   median circuits per job: "
      f"{int(df.groupby('job_id').size().median())}   max: {df.groupby('job_id').size().max()}")

## Where to go next

- `qstd_v1.0_columns.csv` — every column with its group, unit, unit confidence
  and how often it is filled per device.
- `experiments/` — the models behind the paper. `bash scripts/reproduce_all.sh`
  runs them all; about two hours on 8 cores.
- `pipeline/` — stages 00-10, how the dataset was built from raw job records.
  Included to be read, not run: it needs the job database, which is not public.

**Two rules for modelling with this dataset**, both from the paper:

1. **Group your splits by `job_id`.** A job's circuits were submitted together
   and ran back to back; a record-level split puts siblings on both sides and
   inflates every score.
2. **A random split over the whole period rewards recognising the period.** Both
   devices went through distinct campaigns. Split forward in time as well, and
   report both — scores fall substantially when you do.